# 32-candidate generation on a local PC (RTX 3060, 6 GB)

This notebook runs `gen_candidate.py` on your own GPU instead of Colab. It is meant to **share the train split with Colab**: Colab generates one range of articles and this PC generates the rest, at the same time. The two parts are joined afterwards with `merge_candidates.py`.

- **Memory:** on Colab, 32 candidates at batch size 4 peaked at 3.5 GB, so a 6 GB card fits batch size 2 to 4. If a batch runs out of memory, the script splits it in half by itself.
- **Speed:** unknown until step 1 measures it. Expect it to be slower than the Colab L4 (about 3.5 s per article there). On its own, a laptop GPU would need weeks for the 193,883 train articles, which is why the split is shared.
- **Long runs:** the generation keeps running only while this notebook (or the terminal) stays open and the PC stays awake. Set Windows power settings so the PC does not sleep, and keep a laptop plugged in.

Steps: 0 setup (once) → 1 speed test → 2 choose the range → 3 generate → 4 check → 5 merge with the Colab part.

## 0. One-time setup

The Colab conda env (`spec-file.txt`) is for Linux and does not install on Windows, so create a separate env. In **Anaconda Prompt**, from the project folder (`D:\project\BRIO`):

```
conda create -n brio-local python=3.10 -y
conda activate brio-local
pip install torch==2.4.1 --index-url https://download.pytorch.org/whl/cu121
pip install transformers==4.46.3 sentencepiece indobenchmark-toolkit==0.1.7 nltk pandas ipykernel
pip install ./compare-mt
```

Torch is installed first, from the CUDA index, so the later packages don't pull in a CPU-only torch. Then select **brio-local** as the kernel of this notebook (VS Code: kernel picker, top right).

You also need, inside the project folder:
- `indobart-liputan6-finetuned/` with the model files (`config.json`, `pytorch_model.bin` or `model.safetensors`, `sentencepiece.bpe.model`, ...). Download `indobart-liputan6-finetuned.zip` from `MyDrive/BRIO/` and unzip it there. The folder that is there now only holds training logs.
- `liputan6_converted/canonical/train.source` and `train.target` (already there).

In [ ]:
# Check the environment: versions and the GPU.
import sys, torch, transformers
print("python      ", sys.version.split()[0])
print("torch       ", torch.__version__, "| CUDA build:", torch.version.cuda)
print("transformers", transformers.__version__)
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE - check the torch install")
if torch.cuda.is_available():
    print("GPU memory   %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

In [ ]:
# Settings and paths. The notebook lives in colab_vsc/, the scripts in the project folder.
import os
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "colab_vsc" else Path.cwd()
os.chdir(REPO)

DATA_DIR = REPO / "liputan6_converted" / "canonical"
MODEL_DIR = REPO / "indobart-liputan6-finetuned"
OUT_DIR = REPO / "liputan6_candidates_32" / "canonical"     # local output folder
OUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_CANDS = 32
DIVERSITY_PENALTY = 0.5     # must be the same as on Colab
BATCH_SIZE = 2              # 6 GB GPU: start with 2, try 4 after the speed test
BF16 = False                # must be the same as on Colab (see step 1)
N_TRAIN = 193883

print("project folder:", REPO)
for p in [DATA_DIR / "train.source", DATA_DIR / "train.target", DATA_DIR / "val.source", MODEL_DIR / "config.json"]:
    print("found  " if p.exists() else "MISSING", p)

In [ ]:
# Helper: run gen_candidate.py with this kernel's Python and show its output live.
import subprocess, time

# child processes print UTF-8, and their output is captured so it shows up in the notebook
ENV = dict(os.environ, PYTHONIOENCODING="utf-8")

def run_py(args):
    r = subprocess.run([sys.executable] + args, capture_output=True, text=True, encoding="utf-8", errors="replace", env=ENV)
    print(r.stdout + r.stderr)
    return r.returncode

def run_gen(split, out_file, start=0, end=None, num_samples=None, batch_size=None, bf16=None, extra=()):
    cmd = [sys.executable, "-u", "gen_candidate.py", "--dataset", "liputan6",
           "--src_dir", str(DATA_DIR / f"{split}.source"), "--tgt_dir", str(out_file),
           "--model_dir", str(MODEL_DIR), "--num_cands", str(NUM_CANDS),
           "--diversity_penalty", str(DIVERSITY_PENALTY), "--batch_size", str(batch_size or BATCH_SIZE),
           "--start_index", str(start), "--sync_every", "50"]
    if end is not None:
        cmd += ["--end_index", str(end)]
    if num_samples is not None:
        cmd += ["--num_samples", str(num_samples)]
    if (BF16 if bf16 is None else bf16):
        cmd += ["--bf16"]
    cmd += list(extra)
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", env=ENV)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    return proc.returncode, time.time() - t0

def check(out_file, split):
    run_py(["check_candidates.py", "--out_file", str(out_file), "--tgt_file", str(DATA_DIR / f"{split}.target"),
            "--num_cands", str(NUM_CANDS), "--show", "1"])

## 1. Speed test

Generates 32 candidates for the first 24 validation articles, once in full precision and once in bfloat16, and checks the candidates. Read the "s per article" and "peak GPU memory" lines.

- Choose `BF16 = True` only if bfloat16 is clearly faster **and** the check shows no ROUGE = 0 candidates. Use the same choice on Colab (`--bf16` in its command), so the whole train split is generated the same way.
- If peak memory stays well under 6 GB, try `BATCH_SIZE = 4` and rerun this cell to compare.

In [ ]:
# Full precision vs bfloat16 on 24 validation articles (the ROUGE = 0 regeneration is off here, so the raw output is checked).
N_TEST = 24
results = {}
for name, use_bf16 in [("fp32", False), ("bf16", True)]:
    out = REPO / "penalty_test_local" / name / "val.out"
    if out.exists():
        out.unlink()          # start fresh each time, so the timing is real
    print("=====", name)
    code_, seconds = run_gen("val", out, num_samples=N_TEST, bf16=use_bf16, extra=["--no_rouge_check"])
    if code_ == 0:
        check(out, "val")
        results[name] = seconds / N_TEST
for name, s in results.items():
    print("%s: about %.1f s per article (wall clock, including model loading)" % (name, s))

### Optional: decoder cache

Generation is slow mainly because the decoder cache is off: on transformers 4.46.3, IndoBART crashes with a CUDA "device-side assert" when it is on. This cell tries it once on 8 articles. A crash here is expected; if it **does** run, check the candidates for gibberish before using it, because the old 16-candidate files with gibberish at positions 14 and 15 may have come from the same cache problem.

In [ ]:
# Try generation with the decoder cache on (expected to fail on transformers 4.46.3).
out = REPO / "penalty_test_local" / "cache" / "val.out"
if out.exists():
    out.unlink()
code_, seconds = run_gen("val", out, num_samples=8, extra=["--no_rouge_check", "--use_cache"])
if code_ == 0:
    check(out, "val")
    print("ran in %.1f s per article - compare with the speed test above" % (seconds / 8))
else:
    print("cache on failed, as expected - keep it off")

## 2. Choose the range for this PC

Colab keeps writing `train.out` from article 0; this PC takes the end of the split. To finish both at about the same time, give each machine a share proportional to its speed. The cell below works out the split point from the two speeds and from how far Colab already is.

On Colab, add `--end_index {SPLIT}` to the train generation command, so it stops at the split point instead of running to the end. Resuming still works, because the start stays 0.

In [ ]:
# EDIT: the seconds per article measured on each machine, and how many articles Colab has done.
COLAB_S_PER_ARTICLE = 3.5      # from Colab's progress line
LOCAL_S_PER_ARTICLE = 7.0      # from the speed test above
COLAB_DONE = 1760              # "Resuming: N articles" on Colab, or the line count of train.out / 32

remaining = N_TRAIN - COLAB_DONE
colab_rate, local_rate = 1 / COLAB_S_PER_ARTICLE, 1 / LOCAL_S_PER_ARTICLE
SPLIT = COLAB_DONE + round(remaining * colab_rate / (colab_rate + local_rate))
hours = (SPLIT - COLAB_DONE) * COLAB_S_PER_ARTICLE / 3600
print("Colab: articles 0 to %d  (add --end_index %d on Colab)" % (SPLIT - 1, SPLIT))
print("This PC: articles %d to %d (%d articles)" % (SPLIT, N_TRAIN - 1, N_TRAIN - SPLIT))
print("Both finish in about %.0f hours (%.1f days) of running time" % (hours, hours / 24))

## 3. Generate this PC's part

The output goes to `OUT_DIR/train.<start>-<end>.out`, with the regeneration log next to it. The run resumes if it is stopped: run the same cell again (the range must stay the same).

For a run of several days, a terminal is safer than the notebook: closing VS Code or the kernel stops the notebook's process. The cell prints the same command for Anaconda Prompt.

In [ ]:
# This PC's range: from SPLIT to the end of the train split.
START, END = SPLIT, N_TRAIN
PART_FILE = OUT_DIR / f"train.{START}-{END}.out"

terminal_cmd = (f'conda activate brio-local && cd /d "{REPO}" && python gen_candidate.py --dataset liputan6 '
                f'--src_dir "{DATA_DIR / "train.source"}" --tgt_dir "{PART_FILE}" --model_dir "{MODEL_DIR}" '
                f'--num_cands {NUM_CANDS} --diversity_penalty {DIVERSITY_PENALTY} --batch_size {BATCH_SIZE} '
                f'--start_index {START} --end_index {END} --sync_every 50' + (' --bf16' if BF16 else ''))
print("Same command for Anaconda Prompt:\n" + terminal_cmd + "\n")

run_gen("train", PART_FILE, start=START, end=END)

## 4. Check this PC's part

In [ ]:
# Line count and the ROUGE = 0 report for the part generated here.
n_lines = sum(1 for _ in open(PART_FILE, encoding="utf-8"))
print("articles done: %d of %d" % (n_lines // NUM_CANDS, END - START))
# check_candidates compares with the references from the first line, so check a part through a copy
# of the matching reference lines
part_ref = OUT_DIR / f"train.{START}-{END}.target"
with open(DATA_DIR / "train.target", encoding="utf-8") as f:
    refs = f.read().splitlines()[START:END]
part_ref.write_text("\n".join(refs) + "\n", encoding="utf-8")
run_py(["check_candidates.py", "--out_file", str(PART_FILE), "--tgt_file", str(part_ref),
        "--num_cands", str(NUM_CANDS), "--show", "1"])

## 5. Merge with the Colab part

When both parts are finished:

1. Download Colab's `train.out` from `MyDrive/BRIO/liputan6_candidates_32/canonical/` into `OUT_DIR` here, under the name `train.colab.out`.
2. Run the cell below. It checks that the two parts cover all 193,883 articles without gaps or overlaps and that both are complete, then writes `train.out`.
3. Upload the merged `train.out` back to `MyDrive/BRIO/liputan6_candidates_32/canonical/` (replacing Colab's partial file), and continue with step 4 (preprocessing) of `BRIO_32_candidates.ipynb`.

In [ ]:
# Join Colab's part (articles 0 to SPLIT-1) and this PC's part (SPLIT to the end) into one train.out.
COLAB_FILE = OUT_DIR / "train.colab.out"
run_py(["merge_candidates.py", "--num_cands", str(NUM_CANDS), "--num_articles", str(N_TRAIN),
        "--part", f"0:{SPLIT}:{COLAB_FILE}", "--part", f"{SPLIT}:{N_TRAIN}:{PART_FILE}",
        "--out", str(OUT_DIR / "train.out")])